In [0]:
%pip install faker

In [0]:
%run ../gen_datasets

In [0]:
# Definimos una lista de nombres de datasets
# Otra con los nombres de las funciones que generan esos datasets
# Y armamos una lista de tuplas, donde cada tupla es un dataset y su función correspondiente
# La lista de tuplas la genera la función zip de python

datasets = ["equipos", "ordenes", "telemetria"]
funciones = [generar_equipos, generar_ordenes_trabajo, generar_telemetria]
# crear un dict con dataset name como key
datasets_dict = dict(zip(datasets, funciones))

In [0]:
import json 

dataset = dbutils.widgets.get("dataset")
print(f"Obteniedo datos para {dataset}")
data = datasets_dict[dataset]()
print(f"Guardando datos para {dataset}")
dbutils.fs.put(
    f"/Volumes/sandbox/bronze/landing/{dataset}/data.json", 
    json.dumps(data), 
    overwrite=True
    )
    
copy_query = f"""
        COPY INTO sandbox.bronze.{dataset}
        FROM '/Volumes/sandbox/bronze/landing/{dataset}'
        FILEFORMAT = JSON
        FORMAT_OPTIONS (
            'multiline' = 'true',
            'mergeSchema' = 'true'
            )
        COPY_OPTIONS ('mergeSchema' = 'true')
    """
spark.sql(copy_query)